# Open-ended achievements: three planner designs on one budget

The chain notebooks pose one recipe in a carved world. Here the world is an
unmodified Craftax-Classic map and the objective is *"Unlock as many
achievements as you can."* Three designs run on the same skills, executor,
worlds and budget:

1. `saycan_recipe_aware`: SayCan with the recipe-aware VLM prompt as the Say
   term and the competence prior as the Can term.
2. `subgoal_vlm`: the VLM proposes formal sub-goals over the achievements
   its skills unlock, and the symbolic planner follows them, re-asking on low vitals, nearby
   threats and nightfall.
3. `subgoal_scripted`: the same sub-goal planner with a fixed tech-tree
   curriculum (`ScriptedGoalSelector`) in place of the VLM - what the VLM
   has to beat.

Each episode gets 80 skill calls and 3000 environment steps, and ends early
on death or when the planner has nothing left to propose. Every design runs
with creatures off and on, on matched seeds. Scores follow Crafter:
per-achievement success rates and their geometric mean. With seven skills
only seven achievements are reachable at all (collect_wood, place_table,
make_wood_pickaxe, make_wood_sword, collect_stone, make_stone_pickaxe,
collect_drink), so the score ceiling is set by the skill set, and the
designs differ in how reliably and how cheaply they reach it, and in how
they survive.

Start it in a fresh kernel so the JAX memory fraction is applied before JAX
initializes.

In [ ]:
# Install Tiny-SIMA and the optional VLM backend into this kernel.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Leave room for the PyTorch VLM before importing JAX.
os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.35"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,vlm,cuda12" if has_cuda else "notebook,vlm"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

# Import Torchvision before Transformers. Transformers caches optional
# dependency availability on first import.
torch = importlib.import_module("torch")
torchvision = importlib.import_module("torchvision")
transformers = importlib.import_module("transformers")
outlines = importlib.import_module("outlines")
transformers_utils = importlib.import_module("transformers.utils")
if not transformers_utils.is_torchvision_available():
    raise RuntimeError(
        "Transformers cached Torchvision as unavailable. Restart this "
        "kernel, then run this installation cell before any other cell."
    )

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("CUDA detected:", has_cuda)
print("JAX memory fraction:", os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])

In [ ]:
import dataclasses
import json
import time

import jax

from tiny_sima.config import load_vlm_planner_config
from tiny_sima.envs import make_open_ended_environment
from tiny_sima.evaluation import (
    ACHIEVEMENT_NAMES,
    ACHIEVEMENT_TIERS,
    DEFAULT_MAX_ENVIRONMENT_STEPS,
    DEFAULT_MAX_SKILL_CALLS,
    WOOD_TO_STONE_PICKAXE_CHAIN,
    collect_affordance_samples,
    evaluate_open_ended,
    fingerprint,
    fit_affordance_competence_prior,
    git_commit,
    json_ready_open_ended_result,
    load_goal_conditioned_checkpoint,
    make_learned_skill_executor,
    open_ended_planners,
    provenance_mismatch,
    run_provenance,
    with_provenance,
)
from tiny_sima.goals import ALL_SKILL_IDS
from tiny_sima.memory import EpisodeMemory
from tiny_sima.planner import (
    OBSERVATION_PRESETS,
    CompetencePrior,
    TransformersVLMBackend,
    make_critic_affordance_scorer,
)

print("JAX devices:", jax.devices())

In [ ]:
BASELINE_CHECKPOINT = (
    repo_root / "runs" / "skill-ppo-local-1m" / "checkpoints" / "final"
)
# The executor notebook 07 selected, as in notebook 06. The open-ended task
# uses all seven skills, so the executor must know the stone pickaxe.
EXECUTOR_SELECTION_PATH = (
    repo_root / "runs/executor-improvement-stone_pickaxe/selected_executor.json"
)
CANDIDATE_CHECKPOINT = (
    repo_root / json.loads(EXECUTOR_SELECTION_PATH.read_text())["checkpoint"]
    if EXECUTOR_SELECTION_PATH.is_file()
    else None
)
EXECUTOR_CHECKPOINT = CANDIDATE_CHECKPOINT or BASELINE_CHECKPOINT
VLM_CONFIG_PATH = repo_root / "configs" / "vlm_planner_qwen3_2b.yaml"

# The competence prior is fitted on the stone-pickaxe chain, the chain whose
# boundaries cover all seven skills, and shared with notebook 06 when it has
# already been saved there.
PRIOR_CHAIN = WOOD_TO_STONE_PICKAXE_CHAIN
PRIOR_PATH = (
    EXECUTOR_CHECKPOINT.parents[1] / f"competence_prior_{PRIOR_CHAIN.name}.json"
)
REFIT_PRIOR = False
NUM_CALIBRATION_EPISODES = 12
CALIBRATION_SEED = 7

# What the VLM sees besides the frame. The open-ended prompts lean on vitals,
# creatures and unlocked achievements, which the image alone does not state;
# set to None for the image alone, as in notebook 06.
OBSERVATION = OBSERVATION_PRESETS["image_and_state"]

# Small by default: every episode decodes in lockstep with the others, but a
# sub-goal design can ask the VLM up to three times per decision.
NUM_EPISODES = 4
EVALUATION_SEED = 51
SKILL_HORIZON = 64
MAX_SKILL_CALLS = DEFAULT_MAX_SKILL_CALLS  # 80
MAX_ENVIRONMENT_STEPS = DEFAULT_MAX_ENVIRONMENT_STEPS  # 3000
MOB_SETTINGS = (False, True)
DESIGNS = ("saycan_recipe_aware", "subgoal_vlm", "subgoal_scripted")
# Designs reloaded from the newest saved run of this seed instead of rerun, for
# a change that cannot affect them, such as ("saycan_recipe_aware",) while
# iterating on the sub-goal planner. A saved result is reused only when its
# provenance -- executor, VLM configuration, observation, prior, episodes,
# seed and limits -- matches this run's; provenance cannot see code, so rerun
# a design whose own code changed.
REUSE_DESIGNS = ()

RUNS_DIR = repo_root / "runs" / "open-ended"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

vlm_config = load_vlm_planner_config(VLM_CONFIG_PATH)
print("VLM:", vlm_config.model_id)
print("Executor:", EXECUTOR_CHECKPOINT, "exists:", EXECUTOR_CHECKPOINT.exists())
print(f"Episodes: {NUM_EPISODES} per design and setting, seed {EVALUATION_SEED}")

In [ ]:
backend = TransformersVLMBackend(vlm_config)

checkpoint = load_goal_conditioned_checkpoint(EXECUTOR_CHECKPOINT)
executor = make_learned_skill_executor(checkpoint, deterministic=False)
missing = [skill for skill in ALL_SKILL_IDS if skill not in executor.skill_ids]
if missing:
    raise ValueError(
        f"The executor was not trained on {[skill.value for skill in missing]}. "
        "Train a seven-skill executor in notebook 07 and select it."
    )

if PRIOR_PATH.exists() and not REFIT_PRIOR:
    competence_prior = CompetencePrior.load(PRIOR_PATH)
    print("Loaded competence prior:", PRIOR_PATH)
else:
    started = time.perf_counter()
    fit_samples = collect_affordance_samples(
        make_critic_affordance_scorer(checkpoint),
        executor,
        chain=PRIOR_CHAIN,
        num_episodes=NUM_CALIBRATION_EPISODES,
        seed=CALIBRATION_SEED,
        candidate_skills=PRIOR_CHAIN.skills,
        skill_horizon=SKILL_HORIZON,
    )
    competence_prior = fit_affordance_competence_prior(
        fit_samples, skills=PRIOR_CHAIN.skills
    )
    competence_prior.save(PRIOR_PATH)
    print(
        f"Fitted competence prior on {len(fit_samples)} probes in "
        f"{time.perf_counter() - started:.1f}s: {PRIOR_PATH}"
    )

## Closed loop

Each (design, mobs) pair runs the same `NUM_EPISODES` keys, so the worlds
match across designs, and the two mob settings share their maps (the
creatures are the only difference). Planners are rebuilt per setting so no
response cache crosses settings.

In [ ]:
def design_provenance(mobs):
    """Everything a design's outcome depends on, besides code."""

    return run_provenance(
        executor=EXECUTOR_CHECKPOINT,
        vlm=dataclasses.asdict(vlm_config),
        observation=None if OBSERVATION is None else repr(OBSERVATION),
        competence_prior=fingerprint(competence_prior.to_dict()),
        num_episodes=NUM_EPISODES,
        seed=EVALUATION_SEED,
        skill_horizon=SKILL_HORIZON,
        max_skill_calls=MAX_SKILL_CALLS,
        max_environment_steps=MAX_ENVIRONMENT_STEPS,
        mobs=mobs,
    )


def saved_design(design, mobs):
    """Return the newest reusable saved result, or None and why there is none."""

    provenance = design_provenance(mobs)
    reason = "not saved"
    for run in sorted(RUNS_DIR.glob(f"seed{EVALUATION_SEED}-*"), reverse=True):
        path = run / f"{design}_mobs-{'on' if mobs else 'off'}.json"
        if not path.is_file():
            continue
        result = json.loads(path.read_text())
        mismatch = provenance_mismatch(result, provenance)
        if mismatch is None:
            return result, None
        if reason == "not saved":
            reason = f"{mismatch} since {run.name}"
    return None, reason


commit = git_commit(repo_root)
print("commit:", commit)
results = {}
# The commit each reused (design, mobs) result ran at.
reused = {}
for mobs in MOB_SETTINGS:
    environment = make_open_ended_environment(mobs=mobs)
    planners = open_ended_planners(
        ALL_SKILL_IDS, backend, vlm_config, competence_prior, OBSERVATION
    )
    for design in DESIGNS:
        if design in REUSE_DESIGNS:
            saved, mismatch = saved_design(design, mobs)
            if saved is not None:
                results[(design, mobs)] = saved
                reused[(design, mobs)] = saved["commit"]
                summary = saved["summary"]
                print(
                    f"{design:22s} mobs={mobs!s:5s} "
                    f"score={summary['crafter_score']:6.2f} "
                    f"achievements={summary['achievements_per_episode']['mean']:.2f}"
                    f" reused={saved['commit']}"
                )
                continue
            print(f"{design:22s} mobs={mobs!s:5s} rerunning: {mismatch}")
        started = time.perf_counter()
        result = evaluate_open_ended(
            planners[design],
            executor,
            environment,
            num_episodes=NUM_EPISODES,
            seed=EVALUATION_SEED,
            skill_horizon=SKILL_HORIZON,
            max_skill_calls=MAX_SKILL_CALLS,
            max_environment_steps=MAX_ENVIRONMENT_STEPS,
            memory_factory=EpisodeMemory,
        )
        result["design"] = design
        result["seconds"] = time.perf_counter() - started
        results[(design, mobs)] = with_provenance(
            result, design_provenance(mobs), commit
        )
        summary = result["summary"]
        print(
            f"{design:22s} mobs={mobs!s:5s} "
            f"score={summary['crafter_score']:6.2f} "
            f"achievements={summary['achievements_per_episode']['mean']:.2f} "
            f"seconds={result['seconds']:.0f}"
        )

## Metrics

Achievements per episode are mean and standard deviation over episodes.
The Crafter score is `exp(mean_i ln(1 + s_i)) - 1` over all 22
achievements with `s_i` in percent, so an achievement no episode unlocks
pulls it down as much as any other. VLM calls are goal-source calls: the
scripted selector's answers are counted the same way, so its column shows
how often the sub-goal planner re-asks.

In [ ]:
header = (
    f"{'design':22s} {'mobs':5s} {'score':>6s} {'ach mean':>8s} {'ach std':>7s} "
    f"{'skills':>6s} {'steps':>6s} {'vlm':>5s}  terminations"
)
print(header)
print("-" * len(header))
for (design, mobs), result in results.items():
    summary = result["summary"]
    terminations = {
        name: count for name, count in summary["termination_counts"].items() if count
    }
    print(
        f"{design:22s} {mobs!s:5s} {summary['crafter_score']:6.2f} "
        f"{summary['achievements_per_episode']['mean']:8.2f} "
        f"{summary['achievements_per_episode']['std']:7.2f} "
        f"{summary['skill_calls']['mean']:6.1f} "
        f"{summary['environment_steps']['mean']:6.0f} "
        f"{summary['vlm_calls']['mean']:5.1f}  {terminations}"
    )

In [ ]:
# Per-achievement success rates, in tier order; achievements no run unlocked
# are left out to keep the table short.
columns = list(results)
header = " ".join(f"{d[:12]:>12s}/{'on' if m else 'off':3s}" for d, m in columns)
print(f"{'achievement':22s} {header}")
for names in ACHIEVEMENT_TIERS.values():
    for name in names:
        rates = [results[key]["summary"]["success_rates"][name] for key in columns]
        if any(rates):
            print(f"{name:22s} " + " ".join(f"{rate:16.2f}" for rate in rates))

In [ ]:
# Tier breakdown: the mean success rate of each tier's achievements, and how
# many of them an episode unlocks.
print(f"{'tier':14s} {header}")
for tier in ACHIEVEMENT_TIERS:
    cells = []
    for key in columns:
        tier_summary = results[key]["summary"]["tiers"][tier]
        cells.append(
            f"{tier_summary['success_rate']:7.2f} "
            f"({tier_summary['achievements_per_episode']['mean']:.1f})".rjust(16)
        )
    print(f"{tier:14s} " + " ".join(cells))

In [ ]:
# One JSON file per run, beside a manifest of what was run.
stamp = time.strftime("%Y%m%d-%H%M%S")
run_dir = RUNS_DIR / f"seed{EVALUATION_SEED}-{stamp}"
run_dir.mkdir(parents=True, exist_ok=True)
for (design, mobs), result in results.items():
    path = run_dir / f"{design}_mobs-{'on' if mobs else 'off'}.json"
    path.write_text(json.dumps(json_ready_open_ended_result(result), indent=2))
manifest = {
    "commit": commit,
    # Designs reloaded from an earlier run, by the commit each ran at.
    "reused": {
        f"{design}_mobs-{'on' if mobs else 'off'}": reused_commit
        for (design, mobs), reused_commit in reused.items()
    },
    "executor": str(EXECUTOR_CHECKPOINT),
    "vlm": vlm_config.model_id,
    "observation": None if OBSERVATION is None else repr(OBSERVATION),
    "competence_prior": str(PRIOR_PATH),
    "num_episodes": NUM_EPISODES,
    "seed": EVALUATION_SEED,
    "skill_horizon": SKILL_HORIZON,
    "max_skill_calls": MAX_SKILL_CALLS,
    "max_environment_steps": MAX_ENVIRONMENT_STEPS,
    "achievements": list(ACHIEVEMENT_NAMES),
    "tiers": {tier: list(names) for tier, names in ACHIEVEMENT_TIERS.items()},
    "summaries": {
        f"{design}_mobs-{'on' if mobs else 'off'}": result["summary"]
        for (design, mobs), result in results.items()
    },
}
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, default=float))
print("Saved:", run_dir)